In [ ]:
#Bài tập hướng dẫn mẫu AKT

import copy
from heapq import heappush, heappop

n = 3

rows = [1, 0, -1, 0]
cols = [0, -1, 0, 1]

class PriorityQueue:
    def __init__(self):
        self.heap = []

    def push(self, key):
        heappush(self.heap, key)

    def pop(self):
        return heappop(self.heap)

    def empty(self):
        return len(self.heap) == 0

class Node:
    def __init__(self, mats, parent, levels, costs, empty_tile_posi):
        self.parent = parent
        self.mats = mats
        self.levels = levels
        self.costs = costs
        self.empty_tile_posi = empty_tile_posi

    def __lt__(self, nxt):
        return (self.costs + self.levels) < (nxt.costs + nxt.levels)

def calculateCost(mats, final):
    count = 0
    for i in range(n):
        for j in range(n):
            if mats[i][j] and mats[i][j] != final[i][j]:
                count += 1
    return count

def newNodes(mats, empty_tile_posi, new_empty_tile_posi, levels, parent, final):
    new_mats = copy.deepcopy(mats)
    x1, y1 = empty_tile_posi
    x2, y2 = new_empty_tile_posi

    new_mats[x1][y1], new_mats[x2][y2] = new_mats[x2][y2], new_mats[x1][y1]

    costs = calculateCost(new_mats, final)
    return Node(new_mats, parent, levels, costs, new_empty_tile_posi)

def printMatrix(mats):
    for i in range(n):
        for j in range(n):
            print("%d" % mats[i][j], end=" ")
        print()
def isSafe(x, y):
    return 0 <= x < n and 0 <= y < n
def printPath(root):
    if root is None:
        return
    printPath(root.parent)
    printMatrix(root.mats)
    print()

def solve(initial, empty_tile_posi, final):
    pq = PriorityQueue()
    costs = calculateCost(initial, final)
    root = Node(initial, None, 0, costs, empty_tile_posi)
    pq.push(root)

    while not pq.empty():
        minimum = pq.pop()

        if minimum.costs == 0:
            print("Đã tìm thấy đường đi:\n")
            printPath(minimum)
            return

        for i in range(4):
            new_x = minimum.empty_tile_posi[0] + rows[i]
            new_y = minimum.empty_tile_posi[1] + cols[i]

            if isSafe(new_x, new_y):
                new_tile_posi = [new_x, new_y]
                child = newNodes(minimum.mats, minimum.empty_tile_posi, new_tile_posi, minimum.levels + 1, minimum, final)
                pq.push(child)


if __name__ == "__main__":
    initial = [
        [1, 2, 3],
        [5, 6, 0],
        [7, 8, 4]
    ]
    final = [
        [1, 2, 3],
        [5, 8, 6],
        [0, 7, 4]
    ]
    empty_tile_posi = [1, 2]

    solve(initial, empty_tile_posi, final)

Đã tìm thấy đường đi:

1 2 3 
5 6 0 
7 8 4 

1 2 3 
5 0 6 
7 8 4 

1 2 3 
5 8 6 
7 0 4 

1 2 3 
5 8 6 
0 7 4 



In [ ]:
##Bài tập hướng dẫn mẫu A-Star

from collections import deque

class Graph:
    def __init__(self, adjac_list):
        self.adjac_list = adjac_list

    def get_neighbors(self, v):
        return self.adjac_list.get(v, [])

    def h(self, n):
        H = {
            'A': 1,
            'B': 1,
            'C': 1,
            'D': 1
        }
        return H[n]

    def a_star_algorithm(self, start, stop):
        open_lst = set([start])
        closed_lst = set([])
        g = {}
        g[start] = 0

        par = {}
        par[start] = start

        while len(open_lst) > 0:
            n = None
            for v in list(open_lst):
                if n is None or g[v] + self.h(v) < g[n] + self.h(n):
                    n = v

            if n is None:
                print('Path does not exist')
                return None

            if n == stop:
                reconst_path = []
                while par[n] != n:
                    reconst_path.append(n)
                    n = par[n]
                reconst_path.append(start)
                reconst_path.reverse()
                print('Path found: {}'.format(reconst_path))
                return reconst_path

            for (m, weight) in self.get_neighbors(n):
                if m not in open_lst and m not in closed_lst:
                    open_lst.add(m)
                    par[m] = n
                    g[m] = g[n] + weight
                else:
                    if g[m] > g[n] + weight:
                        g[m] = g[n] + weight
                        par[m] = n
                        if m in closed_lst:
                            closed_lst.remove(m)
                            open_lst.add(m)

            open_lst.remove(n)
            closed_lst.add(n)

        print('Path does not exist!')
        return None

adjac_list = {
    'A': [('B', 1), ('C', 3), ('D', 7)],
    'B': [('C', 1), ('D', 5)],
    'C': [('D', 12)]
}

graph1 = Graph(adjac_list)
graph1.a_star_algorithm('A', 'C')

Path found: ['A', 'B', 'C']


['A', 'B', 'C']

In [ ]:
#Bài tập tại lớp AKT

from inspect import currentframe
from IPython.core.interactiveshell import dis
import heapq

N = 4

MOVES = {
    'LÊN':-N,
    'XUỐNG':N,
    'TRÁI':-1,
    'PHẢI':1
}

class Node:
  def __init__(self, state, parent, action, g, h, zero_idx):
    self.state = state
    self.parent = parent
    self.action = action
    self.g = g
    self.h = h
    self.f = g + h
    self.zero_idx = zero_idx
  def __lt__(self, other):
    return self.f < other.f

def get_target_positions(target_state):
  return {val: (i // N, i % N) for i, val in enumerate(target_state) if val != 0}

def calculate_manhattan(state, target_positions):
  distance = 0
  for i, val in enumerate(state):
    if val != 0:
      curr_r, curr_c = i //N, i%N
      target_r, target_c = target_positions[val]
      distance += abs(curr_r - target_r) + abs(curr_c - target_c)
  return distance
def print_matrix(state):
  for i in range(0, N * N, N):
    row = state[i:i+N]
    print(" ".join(f"{val:2}" for val in row))
  print()

def print_path(node):
  path=[]
  while node:
    path.append(node)
    node = node.parent
  path.reverse()
  for step, n in enumerate(path):
    if step == 0:
      print("Trạng Thái Ban Đầu")
    else:
      print(f"Bước {step}: Di chuyển ô trống sang {n.action}")
    print_matrix(n.state)

def solve_15_puzzle(initial_matrix, target_matrix):
  initial_state = tuple(val for row in initial_matrix for val in row)
  target_state = tuple(val for row in target_matrix for val in row)
  target_positions = get_target_positions(target_state)
  zero_idx =initial_state.index(0)
  h_initial = calculate_manhattan(initial_state, target_positions)
  root = Node(initial_state, None, None, 0, h_initial, zero_idx)

  pq = []
  heapq.heappush(pq, root)
  visited = set()
  visited.add(initial_state)

  nodes_explored = 0
  print("Đang giải...")

  while pq:
        current = heapq.heappop(pq)
        nodes_explored += 1

        if current.state == target_state:
            print(f"Tổng số trạng thái đã duyệt: {nodes_explored}")
            print(f"Tổng số bước đi: {current.g}\n")
            print_path(current)
            return

        curr_r, curr_c = current.zero_idx // N, current.zero_idx % N

        for move_name, move_val in MOVES.items():
            if move_name == 'LÊN' and curr_r == 0: continue
            if move_name == 'XUỐNG' and curr_r == N - 1: continue
            if move_name == 'TRÁI' and curr_c == 0: continue
            if move_name == 'PHẢI' and curr_c == N - 1: continue

            new_zero_idx = current.zero_idx + move_val

            state_list = list(current.state)
            state_list[current.zero_idx], state_list[new_zero_idx] = state_list[new_zero_idx], state_list[current.zero_idx]
            new_state = tuple(state_list)

            if new_state not in visited:
                visited.add(new_state)
                h_new = calculate_manhattan(new_state, target_positions)
                child = Node(new_state, current, move_name, current.g + 1, h_new, new_zero_idx)
                heapq.heappush(pq, child)
  print("Không tìm thấy đường giải")

if __name__ == "__main__":
    initial = [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 0, 12],
        [13, 14, 11, 15]
    ]

    final = [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 11, 12],
        [13, 14, 15, 0]
    ]

    solve_15_puzzle(initial, final)

Đang giải...
Tổng số trạng thái đã duyệt: 3
Tổng số bước đi: 2

Trạng Thái Ban Đầu
 1  2  3  4
 5  6  7  8
 9 10  0 12
13 14 11 15

Bước 1: Di chuyển ô trống sang XUỐNG
 1  2  3  4
 5  6  7  8
 9 10 11 12
13 14  0 15

Bước 2: Di chuyển ô trống sang PHẢI
 1  2  3  4
 5  6  7  8
 9 10 11 12
13 14 15  0



In [ ]:
#Bài tập tại lớp A-Star

import heapq

class Graph:
  def __init__(self, adj_list):
    self.adj_list = adj_list

  def get_neighbors(self, node):
    return self.adj_list.get(node, [])

  def heuri(self, node, goal):
    return 0

  def a_star(self, star, goal):
    open_heap = []
    heapq.heappush(open_heap, (0, star))

    g_score = {star:0}
    parents={star: None}
    closed_set = set()

    while open_heap:
      f, current = heapq.heappop(open_heap)
      if current == goal:
        path = []

        while current is not None:
          path.append(current)
          current = parents[current]
        path.reverse()
        return path

      closed_set.add(current)

      for neighbor, weight in self.get_neighbors(current):
        if neighbor in closed_set:
          continue
        duKien_g = g_score[current] + weight
        if neighbor not in g_score or duKien_g < g_score[neighbor]:
          g_score[neighbor] = duKien_g
          f_score = duKien_g + self.heuri(neighbor, goal)
          heapq.heappush(open_heap, (f_score, neighbor))
          parents[neighbor] = current
    return None




adjac_list = {
    'A': [('B', 1), ('C', 3), ('D', 7)],
    'B': [('C', 1), ('D', 5)],
    'C': [('D', 12)],
    'D': []
}

graph = Graph(adjac_list)
path = graph.a_star('A', 'D')
print("Path found:", path)

Path found: ['A', 'B', 'D']
